In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
# the decision tree regressor rsme came out to be 0.0 due to overfitting.
# therefore using cross validation
from sklearn.model_selection import cross_val_score

In [2]:
# 1. load the dataset
housing = pd.read_excel("housing.csv.xlsx")

In [3]:
# 2. Creating startified test set
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

In [4]:
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index].drop("income_cat", axis=1) # we work on this data
    strat_test_set = housing.loc[test_index].drop("income_cat", axis=1) # we keep this data aside

In [5]:
housing = strat_train_set.copy()

In [6]:
# 3. Separate features
housing_labels = housing["median_house_value"].copy()

In [7]:
housing = housing.drop("median_house_value", axis=1)

In [8]:
# 4. Separate numerical and categorical columns
num_attribs = housing.drop("ocean_proximity", axis=1).columns.tolist()

In [9]:
cat_attribs = ["ocean_proximity"]

In [10]:
# 5. Pipelines
# Numerical pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
 
# Categorical pipeline
cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [11]:
# Full pipeline
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])

In [12]:
# 6. Transform the data
housing_prepared = full_pipeline.fit_transform(housing)

In [13]:
# housing_prepared is now a NumPy array ready for training
housing_prepared.shape

(16512, 13)

In [14]:
# 7. Train the model
# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared,housing_labels)
lin_preds = lin_reg.predict(housing_prepared)
# lin_rsme = root_mean_squared_error(housing_labels,lin_preds)
# print(f"The Root Mean Square Error for the Linear regression is {lin_rsme}")
lin_rmses = -cross_val_score(lin_reg, housing_prepared, housing_labels, scoring="neg_root_mean_squared_error",cv = 10)
print(pd.Series(lin_rmses).describe())

count       10.000000
mean     69204.322755
std       2500.382157
min      65318.224029
25%      67124.346106
50%      69404.658178
75%      70697.800632
max      73003.752739
dtype: float64


In [15]:
# Decision Tree Regressor
dec_reg = DecisionTreeRegressor()
dec_reg.fit(housing_prepared,housing_labels)
dec_preds = dec_reg.predict(housing_prepared)
# dec_rsme = root_mean_squared_error(housing_labels,dec_preds)
# print(f"The Root Mean Square Error for the Decision tree is {dec_rsme}")
# some tough lines incoming
dec_rmses = -cross_val_score(dec_reg, housing_prepared, housing_labels, scoring="neg_root_mean_squared_error",cv = 10)
print(pd.Series(dec_rmses).describe())

count       10.000000
mean     68954.456711
std       2414.086838
min      64303.263563
25%      67565.246769
50%      69043.238953
75%      70501.536205
max      72461.964434
dtype: float64


In [16]:
# Random Forest Regressor
random_forest_reg = RandomForestRegressor()
random_forest_reg.fit(housing_prepared,housing_labels)
random_forest_preds = random_forest_reg.predict(housing_prepared)
# random_forest_rsme = root_mean_squared_error(housing_labels,random_forest_preds)
# print(f"The Root Mean Square Error for the Random Forest is {random_forest_rsme}")
random_forest_rmses = -cross_val_score(random_forest_reg, housing_prepared, housing_labels, scoring="neg_root_mean_squared_error",cv = 10)
print(pd.Series(random_forest_rmses).describe())

count       10.000000
mean     49344.411253
std       2211.588069
min      45823.291194
25%      47694.747467
50%      49420.220558
75%      50593.686366
max      52943.400732
dtype: float64
